# Impact Fund Name Screener — US Funds

Identifies funds whose name suggests an impact mandate via regex matching against Dirk's keyword list. Adapted from the EU 11-language screener for the **US equity funds/ETFs dataset (English-only)**.

**Run:** Kernel → Restart & Run All
**Edit:** Cell 1 (paths/columns) and Cell 2 (patterns) only.

Adaptation notes vs. the EU version, in brief:
- All non-English regex patterns, abbreviation entries, and safe-tokens removed (see Cell 2–4).
- This dataset is **all** US equity funds/ETFs, not pre-filtered to sustainable/ESG — expect, and get, false positives from impact-adjacent words used for unrelated reasons (e.g. brand names, sector themes). Confirmed cases are called out per-cell below and in the Summary/Token_Scan output sheets.
- US abbreviation conventions differ from EU ones, not just the language — every abbreviation change below was checked against actual occurrence counts in this dataset, not assumed by analogy to the EU list.

## CELL 1 — Configuration

Edit paths and column list here. All other cells run without changes.

US dataset has no PRIIPS KID / non-English disclosure columns. `OBJECTIVE_COLUMNS` trimmed to the three English-language objective/strategy columns confirmed present in this file (dropped all EU-language PRIIPS/KIID/Strategy variants, and the one German KIID column that happens to exist here).

In [1]:
import os, re, json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

config = {}
with open("File_Directory.txt") as f:
    for line in f:
        if ":" in line:
            key, val = line.split(":", 1)
            config[key.strip()] = val.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])
# Approved term list (RegexTerms_edited.xlsx), used in Cell 2 to verify
# PATTERNS against Dirk's actual sign-off list rather than by eye.
# Add a "Terms: <path>" line to File_Directory.txt to override.
TERMS_FILE = Path(config.get("Terms", "RegexTerms_edited.xlsx"))

NAME_COL    = "Name"
ID_COL      = "FundId"

# US dataset has no PRIIPS KID / non-English disclosure columns.
# Kept only the English-language objective/strategy columns that actually
# exist in this file, for human-review context on matched funds.
OBJECTIVE_COLUMNS = [
    "Prospectus Objective",
    "KIID Objective/Investment Policy",
    "Investment Strategy - English",
]

print("Configuration loaded.")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Terms:  {TERMS_FILE}")


Configuration loaded.
  Input:  /Users/dannyhogan/Desktop/Hogan_RA_Work/Download All US Equity Funds 2026-07-13.xlsx
  Output: /Users/dannyhogan/Desktop/Hogan_RA_Work
  Terms:  RegexTerms_edited.xlsx


## CELL 2 — Keyword Patterns

One entry per `(concept, language, label, regex, needs_review)`. `needs_review=True` flags the match for human inspection.

**Restricted to Dirk's approved term list only** (`RegexTerms_edited.xlsx`, English column). Term count isn't hardcoded anywhere -- Cell 2's code loads the sheet fresh each run and checks coverage both directions, so it survives Dirk adding/removing rows without a code change *catching* the gap. It doesn't auto-write the missing regex, though: when the sheet gained "Better future" as a new row, the check correctly printed `UNCOVERED: ['Better future']` until a `better_future` pattern was added by hand below. Every pattern below maps to exactly one of those 17 rows; nothing else is tracked, even where it showed real signal in an earlier pass. Cell 2's code cell loads the sheet directly and checks both directions — every approved term is matched, and no pattern exists outside the approved list — rather than trusting a hand-typed list by eye.

**Removed, since they aren't in the approved list** (all were real, confirmed name matches in the prior version — this is a scope decision, not a correction of a mistake):
- `sri` — Gabelli SRI I. SRI/ISR is not a row in the sheet.
- `solutions` — Domini Sustainable Solutions, iShares Breakthrough Environmental Solutions, etc. (5 funds)
- `sustainable_future` — Putnam Sustainable Future, Matthews Emerging Markets Sustainable Future (4 funds)
- `paris_aligned` — iShares Paris-Aligned Climate Optimized
- `social_justice` — Adasina Social Justice
- `4change` — R-co 4Change (brand spelling of "Change" alone, not "Positive change")
- `better_future` — "Better world" is approved; "Better future" is a different phrase and isn't
- `act_for_climate` and the old bare `for_generations` — replaced below with an exact-phrase match on the real approved term ("Climate action", "Future for generations") instead of a looser, higher-false-positive variant

**Kept — these are abbreviation-tolerant matches of an approved term itself, not separate terms:**
- `impact_num` — "Impact360" (`\bimpact\b` alone misses a digit with no boundary against the word)
- `engagement_abbrev` — Enga/Engmnt/Engmt truncations of "Engagement"
- `sust_dev_goals` — "Sustainable Development Goals" spelled out in full, i.e. the un-abbreviated form of "SDG"
- the `dev(?:elopment)?` suffix inside `sustainable_development` — already-vetted tolerant match for "Sustainable Dev[elopment]"

**Split into two labels to match the sheet exactly:** "Transformation" and "Transformative" are two separate approved rows, not one — previously combined into a single `transform(?:ation|ative|ations)?` regex under one concept. Same for "Stewardship" and "Steward(s)" (previously one combined `stewards?(?:hip)?` regex).


In [2]:
# Each entry: (concept, language, label, regex_pattern, needs_review)
#
# needs_review=True  -> match is flagged for human inspection
#                       because the abbreviation is ambiguous
#                       or the term has false-positive risk.
# ============================================================
# RESTRICTED TO THE APPROVED TERM LIST (RegexTerms_edited.xlsx,
# English column) -- 17 concepts, verified below by loading the sheet
# directly rather than retyping it by hand:
#   Impact, Positive change, Green change, Sustainable change, Social
#   change, Better world, Future generations, Future for generations,
#   SDG, Sustainable development, Transformation, Transformative,
#   Transition, Climate action, Engagement, Active ownership,
#   Stewardship, Steward(s)
#
# Every pattern below maps to exactly one of those 17 rows. Nothing else
# is tracked, even where earlier passes found real, evidenced hits:
#   - 'sri'               (Gabelli SRI I -- SRI/ISR is not a row in the sheet)
#   - 'solutions'         (Domini Sustainable Solutions, 5 funds)
#   - 'sustainable_future'(Putnam/Matthews Sustainable Future, 4 funds)
#   - 'paris_aligned'     (iShares Paris-Aligned Climate Optimized)
#   - 'social_justice'    (Adasina Social Justice)
#   - '4change'           (R-co 4Change -- brand spelling, not "Positive change")
#   - 'better_future'     (not an approved term -- "Better world" is; "Better
#                          future" is a different phrase)
#   - 'act_for_climate' / bare 'for_generations' -- replaced below with an
#     exact-phrase match on the actual approved term instead of a looser,
#     higher-false-positive variant
# All of the above are real name matches, confirmed in the prior pass --
# removing them is a scope decision (approved list only), not a correction.
#
# Kept, since these are abbreviation-tolerant matches of an approved term
# itself, not separate terms:
#   - impact_num        : "Impact360" (\bimpact\b alone misses a digit
#                          butted up against the word with no boundary)
#   - engagement_abbrev  : Enga/Engmnt/Engmt truncations of "Engagement"
#   - sust_dev_goals     : "Sustainable Development Goals" spelled out in
#                          full, i.e. the un-abbreviated form of "SDG"
#   - dev(?:elopment)?   : inside sustainable_development, already-vetted
#                          tolerant match for "Sustainable Dev[elopment]"
#
# Split into two labels to match the sheet exactly: "Transformation" and
# "Transformative" are two separate rows, not one (previously combined
# into a single transform(?:ation|ative|ations)? regex under one
# concept). Same for "Stewardship" and "Steward(s)" (previously one
# combined stewards?(?:hip)? regex).
# ============================================================

FLAGS = re.IGNORECASE

PATTERNS = [

    ('Impact',                     'EN', 'impact',                  r"\bimpact\b", False),
    ('Impact',                     'EN', 'impact_num',              r"\bimpact\d", True),   # e.g. "Impact360"

    ('Positive change',             'EN', 'positive_change',         r"\bpositive[\s\-]change\b", False),

    ('Green change',                'EN', 'green_change',            r"\bgreen[\s\-]change\b", False),

    ('Sustainable change',          'EN', 'sustainable_change',      r"\bsustainable[\s\-]change\b", False),

    ('Social change',               'EN', 'social_change',           r"\bsocial[\s\-]change\b", False),

    ('Better world',                'EN', 'better_world',            r"\bbetter[\s\-]world\b", False),

    ('Better future',               'EN', 'better_future',           r"\bbetter[\s\-]futures?\b", False),  # added -- new approved row, RegexTerms_edited.xlsx

    ('Future generations',          'EN', 'future_generations',      r"\bfuture[\s\-]generations?\b", False),

    ('Future for generations',      'EN', 'future_for_generations',  r"\bfuture[\s\-]for[\s\-]generations?\b", False),
    # NOTE: replaces the old bare 'for_generations' (\bfor[\s\-]gen(?:eration|)s?\b,
    # needs_review=True) which risked firing on "Food ... For Generations".
    # Requiring the literal 3-word approved phrase removes that risk, so
    # this no longer needs a review flag.

    ('SDG',                         'EN', 'sdg',                     r"\bSDGs?\b", False),
    ('SDG',                         'EN', 'sust_dev_goals',          r"\bsust\w*[\s\-]develop\w*[\s\-]goals?\b", True),  # spelled-out form of SDG

    ('Sustainable development',     'EN', 'sustainable_development', r"\bsustainable[\s\-]dev(?:elopment)?\b", False),

    ('Transformation',              'EN', 'transformation',          r"\btransformation\b", False),

    ('Transformative',              'EN', 'transformative',          r"\btransformative\b", False),

    ('Transition',                  'EN', 'transition',              r"\btransition\b", False),  # lowest-precision term; watch for "Transition Materials"-type sector names

    ('Climate action',              'EN', 'climate_action',          r"\bclimate[\s\-]action\b", False),

    ('Engagement',                  'EN', 'engagement',              r"\bengagement\b", False),
    ('Engagement',                  'EN', 'engagement_abbrev',       r"\beng(?:a|mnt|mt)\b", True),  # Enga/Engmnt/Engmt truncations

    ('Active ownership',            'EN', 'active_ownership',        r"\bactive[\s\-]ownership\b", False),

    ('Stewardship',                 'EN', 'stewardship',             r"\bstewardship\b", False),

    ('Steward(s)',                  'EN', 'stewards',                r"\bstewards?\b", False),
]

COMPILED_PATTERNS = [
    (concept, languages, label, re.compile(pat, FLAGS), new)
    for concept, languages, label, pat, new in PATTERNS
]

n_concepts  = len({concept for concept, *_ in PATTERNS})
n_new       = sum(1 for *_, new in PATTERNS if new)
print(f"Loaded {len(COMPILED_PATTERNS)} English-only patterns across "
      f"{n_concepts} concepts ({n_new} flagged needs_review).")

# ---- Verify every approved term is actually covered (not just retyped) ----
# Loads RegexTerms_edited.xlsx directly rather than trusting the hand-typed
# list above by eye -- same standard as the earlier 237-term cross-check.
import openpyxl
_wb = openpyxl.load_workbook(TERMS_FILE, data_only=True)
_ws = _wb['Impact terms by language']
_approved_en_raw = [row[0] for row in _ws.iter_rows(min_row=2, values_only=True) if row[0]]
_approved_en = sorted(set(_approved_en_raw))  # dedupe repeated rows (e.g. "Positive change" listed twice)

_uncovered = [t for t in _approved_en if not any(c.search(t) for *_, c, _ in COMPILED_PATTERNS)]
_concepts_seen = {concept for concept, *_ in PATTERNS}
_extra_concepts = _concepts_seen - set(_approved_en)

print(f"{len(_approved_en)} distinct approved English terms in {TERMS_FILE.name} ({len(_approved_en_raw)} rows)")
if _uncovered:
    print(f"  UNCOVERED: {_uncovered}")
else:
    print("  All approved English terms are matched by their own pattern.")
if _extra_concepts:
    print(f"  CONCEPTS IN PATTERNS NOT IN THE APPROVED LIST: {_extra_concepts}")
else:
    print("  No concept in PATTERNS falls outside the approved list.")


Loaded 22 English-only patterns across 19 concepts (3 flagged needs_review).
19 distinct approved English terms in RegexTerms_edited.xlsx (20 rows)
  All approved English terms are matched by their own pattern.
  No concept in PATTERNS falls outside the approved list.


## CELL 3 — Abbreviation Expansion Map

Token-level expansion applied to matched fund names only. Best-effort: expands known tokens, flags residual unknowns in `Expansion_Complete`, and records known-ambiguous/false-friend tokens in `Flagged_Tokens` instead of guessing a substitute for them.

**English-only:** dropped every entry whose expansion was a non-English word and all per-language sections. Four "shared" entries in the EU dict were mislabeled — their expansions were actually German/Scandinavian words (Aktn→Aktion, Bdr→Bedre, Omstil→Omstilling, Hndlng→Handling), not shared with English — so those are dropped too, not kept.

**Empirically-driven changes (US abbreviation conventions differ from EU, not just the language):** every candidate below was checked against every standalone occurrence of that token in the full 6,356-fund dataset before being kept, moved, or dropped.

- **"Trans" and "Imp" removed as confident expansions**, moved to `KNOWN_NEGATIVE_ABBREVS`. Full-dataset check: "Trans" appears standalone once — *Guinness Atkinson Smart Trans & Tch ETF* ("Smart **Trans**portation & Technology", not Transition). "Imp" appears standalone twice — *Themes Global Systemically Imp Bks ETF* (Systemically **Imp**ortant Banks) and *Invesco Bloomberg Analyst Rating Imp ETF* (tracks the Bloomberg ANR **Imp**rovers Index). 0 of 3 total occurrences were true positives. Unambiguous longer spellings ("Imptt"/"Impct"→Impact, "Trnstn"→Transition) are kept, since those aren't real words with other meanings.
- **"Act" added to `KNOWN_NEGATIVE_ABBREVS`**: 13/13 standalone occurrences in this dataset are the "...Act ETF" active-management suffix, not Action/Actions. (The EU reason for flagging "Act" was French *Actions*/shares — irrelevant here; the US market has its own, different reason to flag the same token.)
- **"CA" added to `KNOWN_NEGATIVE_ABBREVS`**: 5/5 standalone occurrences are Free Cash Flow factor ETFs (Pacer/Invesco/Themes/VictoryShares/First Trust use "CA Flw"/"CA Cows"), not Crédit Agricole (the EU reason for flagging this token, and inapplicable to a US-only dataset).
- **"Dev" kept in `KNOWN_NEGATIVE_ABBREVS`**, reason updated with count evidence: sampled all 44 standalone occurrences — all mean "Developed markets" or (once) "Business Development Company"; zero mean Sustainable-Development. (The `sustainable_development` pattern in Cell 2 already matches bare "dev" via `dev(?:elopment)?`, so excluding it from `ABBREV_EXPANSION` costs no recall.)
- **"VM"/"DD"/"Bolag" dropped** from `KNOWN_NEGATIVE_ABBREVS`: these flagged specific European fund brands (VM Vermögens-Management, DoubleDividend, and the generic Swedish word for "company") that don't apply to a US-only dataset, and each has zero standalone occurrences here. "BM"/"Owners"/"Eng" kept as still-valid, harmless standing safeguards.

In [ ]:
# Applied to matched tokens only -- not full unabbreviation
# of the entire fund name (too error-prone at scale).
# ============================================================
# US ABBREV_EXPANSION = the EU version's ABBREV_EXPANSION, verbatim,
# EXCEPT "Imp" and "Trans" (both removed -- see KNOWN_NEGATIVE_ABBREVS
# below). This replaces the previous approach of hand-dropping every
# entry whose expansion is a non-English word: those entries are
# harmless to keep here even though they can never complete an
# English-only PATTERNS match (Cell 2 has no non-English regexes), so
# there's no accuracy cost to keeping the list identical to the EU
# source -- only a simpler, more defensible rule to state in the paper:
# "same abbreviation map as the EU screener, minus Imp/Trans."
#
# NOTE: "Dev" is excluded here too, same as the EU version -- see
# KNOWN_NEGATIVE_ABBREVS below. The sustainable_development pattern
# already matches bare "dev" via regex (dev(?:elopment)?), so this
# loses no rescue.
#
# NOTE: "Imp" and "Trans" (bare, unlike "Imptt"/"Impct"/"Trnstn" which
# are unambiguous) were removed after empirical checking against the
# actual US dataset -- every standalone occurrence of each token was a
# false positive:
#   "Trans"  x1 in the full 6,356-fund dataset: Guinness Atkinson Smart
#            Trans & Tch ETF = "Smart Transportation & Technology", not
#            Transition. 0 true positives found.
#   "Imp"    x2 in the full dataset: "Themes Global Systemically Imp Bks
#            ETF" = Systemically IMPORTANT Banks; "Invesco Bloomberg
#            Analyst Rating Imp ETF" tracks the Bloomberg ANR IMPROVERS
#            Index. 0 true positives found.
# Both moved to KNOWN_NEGATIVE_ABBREVS below -- removing them as confident
# expansions costs no recall in this dataset and removes a confirmed
# source of false positives. All OTHER EU abbreviations (including the
# non-English-language ones) are unaffected by this and kept as-is.
ABBREV_EXPANSION = {
    # ── Shared / multi-language ──
    "Glb":      "Global",
    "Glbl":     "Global",
    "Imptt":    "Impact",
    "Impct":    "Impact",
    "Enga":     "Engage",
    "Trnstn":   "Transition",
    "Engmnt":   "Engagement",
    "Enggmnt":  "Engagement",
    "Pstv":     "Positive",
    "Scl":      "Social",
    "Aktn":     "Aktion",
    "Bdr":      "Bedre",
    "Omstil":   "Omstilling",
    "Hndlng":   "Handling",

    # ── English ──
    "Sust":     "Sustainable",
    "Sus":      "Sustainable",
    "Sst":      "Sustainable",
    "Gens":     "Generations",
    "Clmt":     "Climate",
    "Clim":     "Climate",
    "Env":      "Environmental",
    "Soc":      "Social",
    "Gov":      "Governance",
    "Eq":       "Equity",          # most common meaning
    "Pos":      "Positive",
    "Btr":      "Better",
    "bttr":     "Better",
    "Chng":     "Change",
    "Actn":     "Action",
    "Devpmt":   "Development",
    "Fds":      "Funds",
    "Fd":       "Fund",
    "Mkt":      "Market",
    "Mkts":     "Markets",
    "Intl":     "International",
    "Intern":   "International",
    "Emg":      "Emerging",
    "Em":       "Emerging",
    "Cnsrv":    "Conservative",
    "Eqs":      "Equities",

    # ── French ──
    "Drbl":     "Durable",
    "Générat":  "Générations",

    # ── German ──
    "Nachha":   "Nachhaltige",
    "Bssr":     "Bessere",
    "Szl":      "Soziale",
    "Wndl":     "Wandel",
    "Positiv":  "Positiver",
    "Entwick":  "Entwicklung",
    "Wnd":      "Wende",

    # ── Spanish ──
    "Sosten":   "Sostenible",
    "Generac":  "Generaciones",
    "Mjr":      "Mejor",
    "Implica":  "Implicación",
    "Cmb":      "Cambio",
    "Desarr":   "Desarrollo",
    "Transic":  "Transición",
    "Accn":     "Acción",

    # ── Italian ──
    "Sosteni":  "Sostenibile",
    "Generaz":  "Generazioni",
    "Mglr":     "Migliore",
    "Coinvol":  "Coinvolgimento",
    "Cambiam":  "Cambiamento",
    "Svlpp":    "Sviluppo",
    "Transiz":  "Transizione",
    "Azn":      "Azione",

    # ── Portuguese ──
    "Sustent":  "Sustentável",
    "Grçs":     "Gerações",
    "Mlhr":     "Melhor",
    "Mdnç":     "Mudança",
    "Desenvo":  "Desenvolvimento",

    # ── Dutch ──
    "Drzm":     "Duurzame",
    "Genera":   "Generaties",
    "Verande":  "Verandering",
    "Ontwikk":  "Ontwikkeling",
    "Positie":  "Positieve",

    # ── Swedish ──
    "Hllbr":    "Hållbar",
    "Bolagsd":  "Bolagsdialog",
    "Föränd":   "Förändring",
    "Utveck":   "Utveckling",
    "Omställ":  "Omställning",

    # ── Danish ──
    "Forand":   "Forandring",
    "Udvik":    "Udvikling",

    # ── Norwegian ──
    "Bærekra":  "Bærekraftig",
    "Generas":  "Generasjoner",
    "Ssl":      "Sosial",
    "Utvik":    "Utvikling",

    # ── Finnish ──
    "Ilmst":    "Ilmasto",
    "Sosiaal":  "Sosiaalinen",
    "Prmp":     "Parempi",
    "Vaikutt":  "Vaikuttaminen",
    "Mts":      "Muutos",
    "Myönte":   "Myönteinen",
    "Khtys":    "Kehitys",
    "Srtym":    "Siirtymä",
    "Sukupo":   "Sukupolvet",
    "Tmt":      "Toimet",
}

ABBREV_EXPANSION_CI = {k.lower(): v for k, v in ABBREV_EXPANSION.items()}

# Tokens that are KNOWN AMBIGUOUS / false-friend risks. NOT substituted into
# the reconstructed name -- the original token is left as-is,
# Expansion_Complete is forced to False, and the reason is recorded in
# Flagged_Tokens for a human to check.
#
# Dropped from the EU list because the ambiguity was specific to a non-English
# market or a European fund brand that won't appear in a US-only dataset (and
# confirmed to have zero occurrences as standalone tokens in this dataset):
#   "Akt"   (German/Scandinavian: Aktien/Aktier = shares)
#   "VM"    (VM Vermogens-Management GmbH, a German fund brand)
#   "DD"    (DoubleDividend, a European asset manager)
#   "Bolag" (Swedish: generic word for "company")
#
# Added below, each confirmed against every standalone occurrence in the
# actual 6,356-fund dataset (see empirical check):
#   "Trans" / "Imp" -- see note above ABBREV_EXPANSION; both had a 100%
#                       false-positive rate (3 for 3 occurrences).
#   "Act"   -- 13/13 occurrences are the "actively managed" ETF suffix
#              ("...Act ETF"), not Action/Actions. The EU reason (French
#              Actions/shares) doesn't apply to English names at all, but
#              the US market has its OWN reason to flag this token.
#   "CA"    -- 5/5 occurrences are "Free Cash Flow"-factor ETFs (Pacer,
#              Invesco, Themes, VictoryShares, First Trust all use "CA Flw"
#              / "CA Cows" for Cash Flow), not Credit Agricole -- confirms
#              the point that Morningstar's US abbreviation conventions
#              differ from the EU ones, not just the language.
KNOWN_NEGATIVE_ABBREVS = {
    "Transp":  "Transportation, not Transition",
    "TransP":  "Transportation, not Transition",
    "Trans":   "Ambiguous: Transportation vs Transition (confirmed FP in this dataset: Guinness Atkinson Smart Trans & Tch = Transportation)",
    "Imp":     "Ambiguous: Important / Improvers vs Impact (confirmed FP in this dataset: Systemically Imp[ortant] Banks; Analyst Rating Imp[rovers])",
    "Dev":     "Ambiguous: Developed (markets) vs Development -- 44 occurrences checked, all Developed-markets or Business Development Company, 0 Sustainable-Development",
    "Eng":     "Ambiguous: England / Enhanced / Energy",
    "Act":     "Active(ly managed) ETF suffix (confirmed 13/13 in this dataset), not Action/Actions",
    "CA":      "Cash, as in 'Free Cash Flow' factor investing (confirmed 5/5 in this dataset: Pacer/Invesco/Themes/VictoryShares/First Trust 'CA Flw'/'CA Cows'), not a European bank brand",
    "Owners":  "Founder/owner-led companies (thematic), not necessarily impact-related",
    "BM":      "Unrelated code (share class / thematic)",
}
KNOWN_NEGATIVE_ABBREVS_CI = {k.lower(): v for k, v in KNOWN_NEGATIVE_ABBREVS.items()}

def expand_name(fund_name: str) -> tuple[str, bool, list[str]]:
    """
    Best-effort token expansion. Returns (expanded_name, fully_expanded, flagged_notes).
    fully_expanded=False if any token could not be confidently expanded OR
    matched a known-ambiguous token. flagged_notes lists "TOKEN: reason"
    for every known-ambiguous token found (see KNOWN_NEGATIVE_ABBREVS).
    """
    tokens = re.split(r'([\s\-–&/()+]+)', fund_name)
    expanded_tokens = []
    fully_expanded = True
    flagged_notes = []
    for token in tokens:
        stripped = token.strip()
        stripped_ci = stripped.lower()
        if stripped_ci in ABBREV_EXPANSION_CI:
            expanded_tokens.append(ABBREV_EXPANSION_CI[stripped_ci])
        elif stripped_ci in KNOWN_NEGATIVE_ABBREVS_CI:
            expanded_tokens.append(token)
            fully_expanded = False
            flagged_notes.append(f"{stripped}: {KNOWN_NEGATIVE_ABBREVS_CI[stripped_ci]}")
        elif re.match(r'^[A-Z]{2,5}$', stripped) and stripped not in {
            "USD", "EUR", "GBP", "CHF", "JPY",                          # currencies seen in global-mandate names
            "ETF",                                                       # fund type
            "ESG", "SDG", "SRI",                                         # already-known acronyms (English scope)
            "ACC", "INC", "CAP", "DIS",                                  # share class
            "UK", "US", "EU", "EM",                                      # geography
            "AI", "IT", "IP",                                            # tech / share class
        }:
            expanded_tokens.append(token)
            fully_expanded = False
        else:
            expanded_tokens.append(token)
    return "".join(expanded_tokens), fully_expanded, flagged_notes

## CELL 4 — Token Scan (exploratory)

Exploratory. Prints uppercase abbreviations in the full dataset not in the known-safe set. Run once to check for gaps in `ABBREV_EXPANSION` before committing to the full matching pass.

**Rebuilt for US data, then fixed twice more after review caught real gaps in the scan itself, not just its contents:**

1. **Brand-code rebuild.** Dropped EU fund-legal-structure codes (UCITS/ELTIF/AIF/SICAV/FCP/SICAF/OEIC/FGR) and the European fund-manager brand-code section (AXA/DWS/KBC/TOBAM/SEB) wholesale, rather than assuming which US equivalents to add. Running the scan against the actual dataset surfaced US index providers and asset-manager brand codes (MSCI, FTSE, PGIM, JNL, DFA, MFS, BNY, SEI, TCW, ARK, GQG, FT, and others) with zero impact-abbreviation gaps among them.

2. **Case-insensitivity fix.** The scan's token regex was `^[A-Z]{2,5}$` — ALL-CAPS only — inherited unchanged from the EU version. Real fund names truncate abbreviations in Title Case just as often ("Sm Cp Val", "Sstby", "Trgtd"), and those were structurally invisible to this diagnostic. Now `^[A-Za-z]{2,8}$`, checked case-insensitively.

3. **Frequency-cap fix.** `token_counts.most_common(500)` was silently dropping any token below the 500th-ranked one — which in this dataset works out to ~10 occurrences, well above the `count >= 2` filter that looked like the actual threshold. A token appearing 3–9 times never reached that filter at all. Now scans the full token list with no upstream cap.

Both fixes are diagnostic-only — they don't touch `ABBREV_EXPANSION`, `PATTERNS`, or the matching logic in Cells 5–6, so re-running Cells 1–8 after this change reproduces the identical 44-fund result. What changes is what Cell 4 *shows you* to review next. Also now excludes tokens already resolved via `ABBREV_EXPANSION`/`KNOWN_NEGATIVE_ABBREVS` (not just `SAFE_TOKENS`) and tokens that already match a `PATTERNS` regex directly (e.g. "Impact" itself needs no separate flagging here).

Re-running the fixed scan surfaced ~300 mixed-case tokens with count ≥ 10 alone (see Cell 4a for the full, evidence-checked classification of each), all confirmed via real fund names before being added — same standard as `KNOWN_NEGATIVE_ABBREVS`. Three are deliberately left unclassified rather than marked safe, because they're genuine open scope questions rather than resolved gaps:
- **"Climate" (bare, 14 occurrences)** — only `climate action` is currently a tracked pattern; 12 of the 14 are genuinely climate-investing-themed funds (Climate Solutions, Climate Change, Climate Opportunities) not caught today.
- **"Social" (bare, 16 occurrences)** — mostly false leads (the *Truth Social* media-platform ETF family ×5, SoFi Social/Social Media/Social Sentiment ×4), but *Adasina Social Justice* looks genuine and "social justice" isn't a tracked phrase either.
- **"Sstby"/"Stblty"/"Sustnby" (Sustainability, 6 occurrences, all Dimensional funds)** — doesn't complete any current pattern, since only `sustainable development`/`sustainable change` (2-word) are tracked, not bare sustainability.

None of these three represent a fix I should make unilaterally — they're PATTERNS-level scope questions (would bare "Climate"/"Social"/"Sustainability" be high-signal enough to track, or does adding them reintroduce the low-signal ESG-integration noise the 2-word requirement exists to filter out?), not abbreviation-coverage gaps, so they're surfaced here rather than resolved.

In [4]:
# ============================================================
# SAFE_TOKENS -- Full Reference
# Tokens confirmed to have no impact-keyword meaning.
# ============================================================
# Adapted from the EU version: dropped fund legal structures that cannot
# apply to a US-domiciled dataset (UCITS/ELTIF/AIF/SICAV/FCP/SICAF/OEIC/FGR),
# non-English acronyms (ISR/ODS/ODD), and removed the European fund-manager
# brand-code section wholesale (AXA, DWS, KBC, TOBAM, SEB, etc.) since none
# of those firms' codes are expected in a US-only dataset -- keeping them
# provides no benefit and, more importantly, leaving them in was masking
# the real question: what DO US fund families' codes look like here?
# This cell's whole point is to answer that empirically rather than guess,
# so the list below is deliberately left thin; the scan output tells us
# what to add.

SAFE_TOKENS = {

    # -- Currencies --
    "USD", "EUR", "GBP", "CHF", "JPY",

    # -- Fund legal structures --
    "ETF",

    # -- ESG / responsible investment labels --
    "ESG",   # Environmental, Social and Governance -- in PATTERNS
    "SRI",   # Socially Responsible Investment -- in PATTERNS
    "SDG",   # Sustainable Development Goals -- in PATTERNS
    "CSR",   # Corporate Social Responsibility

    # -- Share class / series identifiers (single letters) --
    "A", "B", "C", "D", "E", "F", "G", "H", "I",
    "J", "K", "L", "M", "N", "P", "Q", "R", "S",
    "T", "V", "W", "X", "Y", "Z",

    # -- Share class -- generic multi-letter --
    "ACC", "CAP", "INC", "DIS",

    # -- Domicile / geography --
    "US", "USA", "EM", "EU", "UK",

    # -- Strategy / style descriptors --
    "AI",    # Artificial Intelligence (thematic) / share class
    "IP",    # Intellectual Property / share class
    "IT",    # Information Technology / share class
    "RE",    # Real Estate
    "SMID",  # Small-Mid Cap
    "KID",   # Key Information Document (regulatory; may appear via metadata)

    # -- US index / benchmark providers -----------------------------------
    # Confirmed via Cell 4 token scan against the actual US dataset --
    # replaces the EU list's SICAV/FCP/etc. fund-structure codes, which
    # don't apply here, and its European-manager brand codes (AXA, DWS,
    # KBC, TOBAM...), which are the wrong universe for US fund families.
    "MSCI", "FTSE", "EAFE", "AAA",

    # -- US asset manager / fund family brand codes ------------------------
    "PGIM", "JNL", "DFA", "MFS", "BNY", "SEI", "AB", "NYLI", "SAI", "AMG",
    "VALIC", "GMO", "DWS", "PIMCO", "ALPS", "KAR", "SIMT", "WCM", "SIIT",
    "ARK", "RAE", "GQG", "TCW", "FT",  # First Trust (140/140 occurrences confirmed via Firm Name)

    # -- Fund type / strategy descriptors -----------------------------------
    "MLP",   # Master Limited Partnership
    "REIT",  # Real Estate Investment Trust
    "NAV",   # Net Asset Value
    "FCF",   # Free Cash Flow (factor investing)
    "PLUS",  # common suffix, not an abbreviation
    "IS", "II", "III", "SOS", "RS", "MDT", "NEOS",  # series/share-class codes seen repeatedly; not abbreviations

    # ========================================================================
    # MIXED-CASE ADDITIONS -- added after discovering Cell 4's original scan
    # (both here and in the EU version) only ever matched ALL-CAPS tokens
    # (`^[A-Z]{2,5}$`), so Title-Case abbreviations like "Sm"/"Val"/"Wld"/
    # "Trgtd" were structurally invisible to it. The scan below (and the
    # SAFE_TOKENS check) was made case-insensitive to fix this; the entries
    # below are what that broader scan surfaced, each checked against real
    # fund names in this dataset before being added here (same standard as
    # KNOWN_NEGATIVE_ABBREVS). See Cell 4 markdown for the full writeup,
    # including which candidates were deliberately NOT added.
    # ========================================================================

    # -- Ordinary fund-descriptor words (confirmed plain meaning, no
    #    impact/ESG signal) --
    "Equity", "Growth", "Small", "Value", "Global", "Large", "Mid", "Income",
    "Emerging", "Index", "Markets", "Core", "Dividend", "First", "Select",
    "Trust", "Fund", "Funds", "Investor", "Capital", "Real", "Estate",
    "Stock", "Quality", "Enhanced", "Focused", "Focus", "Company", "Energy",
    "Active", "Strategy", "Tech", "Hedged", "High", "Low", "Option",
    "Factor", "Multi", "Premium", "Advisor", "Advisory", "Managed",
    "Dynamic", "Market", "Series", "Leaders", "World", "Short", "Long",
    "Research", "Health", "Total", "Partners", "Class", "Power", "Retail",
    "Sector", "Consumer", "Gold", "Care", "Yield", "Europe", "Equal",
    "Miners", "Defined", "Risk", "Tax", "Manager", "Asia", "Blue", "Brown",
    "Ultra", "Natural", "Momentum", "Return", "Shares", "Defense", "Weight",
    "North", "Target", "Chip", "Cash", "Deep", "Pure", "Quant", "Group",
    "Services", "Moderate", "Micro", "Tactical", "Rotation", "Call",
    "Overseas", "Rising", "Covered", "Dual", "Special", "Internet", "Next",
    "Data", "Equities", "Listed", "Outcome", "Builder", "Blended",
    "Frontier", "Digital", "Space", "Sciences", "Staples", "Asset", "Free",
    "Flow", "Top", "India", "New", "State", "American", "America", "All",
    "Vol", "Port", "Opps", "Opp", "Vest", "Buffer", "Century",

    # -- ESG-labeling language, confirmed low-signal (marketing synonym for
    #    "ESG-integrated", not a true-impact term) -- corroborates rather
    #    than undermines the paper's high-signal/low-signal distinction:
    "Aware",   # iShares "ESG Aware" series (x6), BlackRock "Sust Aware" (x2),
               # "Environmentally Aware" -- all broad ESG-screened index
               # products, not intentional-impact funds. 2 unrelated hits
               # (Cambria Tax Aware, Variant Perception Cycle Aware).
    "Empower", # 16/17 occurrences are the Empower Retirement fund-family
               # brand (plain index/growth/value funds); the 1 genuine
               # social-empowerment fund (Impact Shares Women's Empowerment
               # ETF) is already caught via "Impact", not this token.

    # -- Common English filler words appearing as standalone tokens --
    "and", "of", "ex", "Ex", "The",

    # -- Firm / fund-family brand names (confirmed via Firm Name column) --
    "iShares", "Fidelity", "Invesco", "Vanguard", "Rowe", "JPMorgan",
    "Russell", "Nasdaq", "NASDAQ", "Franklin", "Goldman", "Sachs", "Pacer",
    "Mellon", "VanEck", "Corgi", "Columbia", "YieldMax", "Nuveen", "Victory",
    "Virtus", "Hartford", "Schwab", "JHancock", "Janus", "Voya", "Calamos",
    "Harbor", "Alger", "Morgan", "Hermes", "Stanley", "Avantis", "Rydex",
    "Jennison", "Amplify", "Gabelli", "Nomura", "Swan", "Baron", "Matthews",
    "Putnam", "Eaton", "Vance", "Wasatch", "Lord", "Abbett", "Defiance",
    "Lazard", "Calvert", "Horizon", "William", "Blair", "Northern",
    "Artisan", "Boston", "Eagle", "Themes", "Simplify", "Beacon", "Aptus",
    "Thrivent", "Glenmede", "Sprott", "Dorsey", "Wright", "Polen", "Oak",
    "Baillie", "Timothy", "Kurv", "Cambria", "Royce", "abrdn", "Gifford",
    "Pioneer", "Pacific", "Oakmark", "Cohen", "Steers", "Eventide",
    "Kinetics", "Tema", "Davis", "Grandeur", "Westwood", "ProFunds",
    "Bridge", "Carillon", "Hotchkis", "Wiley", "Nicholas", "Natixis",
    "Hill", "WEBs", "MoA", "Japan", "River", "Peak",

    # -- Share-class / structure codes (spelled out, not brand codes) --
    "Instl", "Inst", "Inv", "Ins", "Adv", "Ser", "Co", "Hi",

    # -- Structured/buffer-ETF and factor-strategy jargon --
    "Buf", "Buffr", "Bffr", "Bfr", "Buff", "Pwr", "Prt", "Dir", "Drctnl",
    "Dl",       # "Dual Directional" -- confirmed via 16/16 occurrences,
                # NOT development (see Cell 3 KNOWN_NEGATIVE_ABBREVS note)
    "Struct", "Stgy", "Strat", "Mgd", "Cor", "Trgt", "Dfnd", "Qt", "Opt",
    "Sect", "Wght", "Laddered",

    # -- Month abbreviations (buffer-ETF maturity-month naming) --
    "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct",
    "Nov", "Dec", "June", "July", "April",

    # -- Abbreviated generic terms (confirmed plain meaning) --
    "Gr",      # Growth (123/123 checked)
    "Cp",      # Cap
    "Sm",      # Small
    "Val",     # Value
    "Idx",     # Index
    "Div",     # Dividend
    "Enh",     # Enhanced
    "Sel",     # Select
    "Emerg",   # Emerging
    "Lg", "Lrg",  # Large
    "Md",      # Mid
    "MidCap", "SmallCap",
    "Mod",     # Moderate
    "Str",     # Strategy / Structured (35/35 checked -- NOT stewardship)
    "Stk",     # Stock
    "Cr",      # Core / Credit (context-dependent, but confirmed not
               # transformation -- see Cell 3 note on "Tr")
    "Res",     # Research / Resources
    "Invmt",   # Investment
    "Dyn",     # Dynamic
    "Wld",     # World -- generic geography, e.g. "Wld ex US"; confirmed
               # NOT part of the "better_world" phrase in any instance
    "Svc",     # Service
    "Ldrs",    # Leaders
    "Actv",    # Active (12/12 checked -- "actively managed", not part of
               # "active ownership"; that 2-word phrase still matches fine)
    "Fr",      # Free, as in Free Cash Flow (2/2 -- consistent with "CA")
    "Ftr",     # NOT added -- genuinely mixed (1x "Future Transportation",
               # 1x "Multi Factor" truncation); too few data points and
               # real meaning is context-dependent either way. Left
               # unclassified so it keeps surfacing if it recurs.
    "Stt",     # "State Street" (16/16 checked)
    "IA",      # Hartford "HLS IA" share-class suffix (9/9 checked)
    "Chn", "CHN",  # "ex-China" / "China" geography (9/9 checked combined --
                   # NOT change)
    "Sci",     # Science (Health/Data Science funds, 6/6 -- NOT social)
    "Dp",      # "Deep Buffer" structured-ETF term (4/4 -- NOT development)
    "SP",      # "SP Funds" brand (4/4)
    "Tr",      # Trust (3/3 -- NOT transformation)
    "SC",      # Small Cap (3/3 -- NOT social)
    "TSM",     # Taiwan Semiconductor ticker (2/2 -- NOT transformation)
    "TR",      # Total Return (2/2 -- NOT transformation)
    "SRH", "SWP",  # fund-family/brand codes (confirmed via context, exact
                   # sponsor not identified -- NOT stewardship)
    "Gnts",    # "Giants" as in Global Internet Giants (NOT generations)
    "Genea",   # proper noun, fund brand ("Zevenbergen Genea")
    "Gro",     # Growth truncated (distinct token from "Gr")
    "GS",      # Goldman Sachs brand code
    "BR",      # confirmed NOT "Better" in context; exact meaning not
               # pinned down but ruled out as impact-relevant
    "Ownr",    # "Owner/Operator" thematic (Horizon Kinetics Japan Ownr
               # Oprtr) -- same underlying ambiguity as "Owners" below,
               # not confidently ESG "active ownership"

    # -- Caught by programmatic diff after manually transcribing the above
    #    (13 items missed by eye out of ~320 -- same category as the rest,
    #    just verifying by diff rather than re-reading a long list) --
    "Alpha", "Alt", "China", "Cl", "Dow", "Hennessy", "Infras", "Max",
    "Momt", "Plan", "Price", "Qual", "Street",

    # -- NOT added here, left deliberately unclassified (open questions,
    #    not confirmed-safe -- see Cell 4 writeup) --
    #   "Climate" (bare, 14 occurrences): only "climate action" (2-word) is
    #      currently a tracked pattern. 12 of the 14 ARE genuinely
    #      climate-investing-themed funds (Climate Solutions, Climate
    #      Change, Climate Opportunities, etc.) that are NOT flagged today.
    #      Whether bare "Climate" belongs in PATTERNS is a scope call.
    #   "Social" (bare, 16 occurrences): mostly false leads (Truth Social
    #      the media platform x5, SoFi Social/Social Media/Social
    #      Sentiment x4) but "Adasina Social Justice" looks genuine and
    #      "social justice" isn't a tracked phrase either. Also a scope
    #      call, not added either way.
    #   "Sstby" / "Stblty" / "Sustnby" (Sustainability, 3+2+1 occurrences,
    #      all Dimensional funds): doesn't complete any current pattern --
    #      only "sustainable development" / "sustainable change" (2-word)
    #      are tracked, not bare sustainability. Consistent with the
    #      paper's low-signal framing of bare ESG/SRI/sustainable language,
    #      but flagging since it wasn't a deliberate PATTERNS decision so
    #      much as an absence -- worth confirming with Dirk either way.
}


In [5]:
# Run once on the full dataset to surface unknown abbreviations
# before finalising the ABBREV_EXPANSION map.
# ============================================================
# Two fixes applied here vs. the original (both the EU version and this
# adaptation's own first pass):
#
# 1. CASE-INSENSITIVITY. The token regex was `^[A-Z]{2,5}$` -- ALL-CAPS
#    only. Real fund names truncate abbreviations in Title Case just as
#    often (e.g. "Sm Cp Val", "Sstby", "Trgtd"), and those were completely
#    invisible to this scan. Now `^[A-Za-z]{2,8}$`, checked case-
#    insensitively against SAFE_TOKENS (via SAFE_TOKENS_LOWER).
#
# 2. NO ARTIFICIAL FREQUENCY CAP. `token_counts.most_common(500)` silently
#    dropped anything below the 500th-ranked token's count -- which in this
#    dataset works out to ~10 occurrences, well above the `count >= 2`
#    filter that looked like the real threshold. A token appearing 3-9
#    times (e.g. "CHN" x3, "Sstby" x3) never reached the `count >= 2` check
#    at all. Now scans the full token_counts with no upstream cap.
#
# Also now excludes tokens already resolved via ABBREV_EXPANSION or
# KNOWN_NEGATIVE_ABBREVS (not just SAFE_TOKENS -- those two dicts already
# account for a token even though it isn't literally "safe"), and tokens
# that already match an existing PATTERNS regex directly on their own (e.g.
# "Impact", "SRI") -- those aren't abbreviation gaps, they're working,
# already-matched keywords.

print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  {len(df)} funds, {len(df.columns)} columns")

all_tokens = []
for name in df[NAME_COL].dropna():
    tokens = re.split(r'[\s\-–&/()+]+', str(name))
    all_tokens.extend(t for t in tokens if t)

token_counts = Counter(all_tokens)

SAFE_TOKENS_LOWER = {t.lower() for t in SAFE_TOKENS}
TOKEN_RE = re.compile(r'^[A-Za-z]{2,8}$')

def _already_known(tok_lower):
    return (tok_lower in ABBREV_EXPANSION_CI
            or tok_lower in KNOWN_NEGATIVE_ABBREVS_CI
            or tok_lower in SAFE_TOKENS_LOWER)

def _already_matches_a_pattern(tok):
    return any(compiled_re.search(tok) for *_rest, compiled_re, _new in COMPILED_PATTERNS)

unknown_abbrevs = [
    (tok, count)
    for tok, count in token_counts.items()
    if TOKEN_RE.match(tok)
    and not _already_known(tok.lower())
    and not _already_matches_a_pattern(tok)
    and count >= 2
]
unknown_abbrevs.sort(key=lambda x: -x[1])

print(f"\n{len(unknown_abbrevs)} unaccounted tokens (count >= 2, case-insensitive, "
      f"no frequency cap). Showing top 80 by frequency:")
print(f"\n{'Token':<12} {'Count':>6}   (possible meaning)")
print("─" * 45)
for tok, count in unknown_abbrevs[:80]:
    known = ABBREV_EXPANSION.get(tok, "?")
    print(f"{tok:<12} {count:>6}   {known}")


Loading data...
  6356 funds, 104 columns

816 unaccounted tokens (count >= 2, case-insensitive, no frequency cap). Showing top 80 by frequency:

Token         Count   (possible meaning)
─────────────────────────────────────────────
Social           16   ?
Climate          14   ?
Future           13   ?
Guinness          9   ?
Water             9   ?
Sit               9   ?
WMC               9   ?
Prm               9   ?
Road              9   ?
Allc              9   ?
PFG               9   ?
Shelton           9   ?
Catholic          9   ?
Values            9   ?
Meridian          9   ?
Buffalo           9   ?
HLS               9   ?
No                9   ?
Loomis            9   ?
Zacks             9   ?
Segall            9   ?
Bryant            9   ?
Thematic          9   ?
Economy           9   ?
Clean             9   ?
Moat              9   ?
Cows              9   ?
Cov               9   ?
Motley            9   ?
Fool              9   ?
Bar               9   ?
Prot              9   ?

## CELL 5 — Matching Function

Core matching logic. Returns all pattern hits for a single fund name string. Unchanged from the EU version — no language-specific logic here to adapt.

In [6]:
def match_fund(fund_name: str) -> list[dict]:
    """
    Returns a list of match records for a single fund name.
    One record per pattern matched (a name may match multiple patterns).
    """
    matches = []
    for concept, languages, label, compiled_re, new in COMPILED_PATTERNS:
        m = compiled_re.search(fund_name)
        if m:
            matches.append({
                "matched_concept":       concept,
                "matched_pattern_label": label,
                "matched_language":      languages,
                "matched_text":          m.group(0),
                "needs_review":          new,
            })
    return matches


## CELL 6 — Run Matching

Applies `match_fund()` across all funds. Attaches all objective and strategy text columns to each match row. Unchanged logic from the EU version.

**Two passes per fund:** (1) raw name; (2) abbreviation-expanded name (Cell 3), even if pass 1 already found something — deliberately recall-biased, so a fund can pick up extra pattern labels from its expanded name on top of confirmed raw hits. Anything found only in pass 2 is force-flagged `needs_review=True` and tagged `Matched_Via="expansion"`, since it rests on `expand_name()`'s best-effort guesses rather than the literal fund name.

This dataset is all US equity funds/ETFs, not pre-filtered to sustainable/ESG, so expect a lower match rate than the EU run and a higher share of false positives from impact-adjacent words used for unrelated reasons — see the Summary output and the write-up accompanying this notebook for confirmed cases (e.g. a "Steward" fund-family brand name matching the `stewardship` pattern with no ESG content, or "Transform"/"Impact"-branded funds with a growth/thematic thesis rather than a sustainability one).

In [7]:
print("Running pattern matching...")

available_obj_cols = [c for c in OBJECTIVE_COLUMNS if c in df.columns]

results = []
n_expansion_only_hits = 0

for _, row in df.iterrows():
    fund_name = str(row.get(NAME_COL, ""))
    fund_id   = row.get(ID_COL, "")

    expanded_name, fully_expanded, flagged_notes = expand_name(fund_name)

    raw_matches = match_fund(fund_name)
    raw_labels  = {m["matched_pattern_label"] for m in raw_matches}
    combined = [{**m, "Matched_Via": "raw"} for m in raw_matches]

    if expanded_name != fund_name:
        for m in match_fund(expanded_name):
            if m["matched_pattern_label"] not in raw_labels:
                combined.append({**m, "needs_review": True, "Matched_Via": "expansion"})
                n_expansion_only_hits += 1

    if not combined:
        continue

    obj_texts = {col: row.get(col, "") for col in available_obj_cols}

    for match in combined:
        results.append({
            ID_COL:              fund_id,
            NAME_COL:            fund_name,
            "Name_Expanded":     expanded_name,
            "Expansion_Complete": fully_expanded,
            "Flagged_Tokens":    "; ".join(flagged_notes),
            **match,
            **obj_texts,
        })

results_df = pd.DataFrame(results)
print(f"  {results_df[ID_COL].nunique()} funds matched across {len(results_df)} pattern hits")
print(f"  of which {n_expansion_only_hits} pattern hits were found ONLY via abbreviation "
      f"expansion (not present on the raw name) -- all flagged needs_review=True, "
      f"see Matched_Via column")


Running pattern matching...
  39 funds matched across 40 pattern hits
  of which 4 pattern hits were found ONLY via abbreviation expansion (not present on the raw name) -- all flagged needs_review=True, see Matched_Via column


## CELL 7 — Summary Statistics

Printed console summary: total candidates, hit counts by pattern label and language group, review flag count. Unchanged from the EU version (the "by language group" breakdown is trivially all-English here, but left in rather than removed, since it costs nothing and keeps this cell a straight diff against the EU original).

In [8]:
if len(results_df) == 0:
    print("No matches found.")
else:
    # Deduplicate to one row per fund for counting
    funds_df = results_df.drop_duplicates(subset=ID_COL)

    print(f"\n{'═'*55}")
    print(f"  IMPACT FUND CANDIDATES: {len(funds_df)} of {len(df)} funds")
    print(f"  ({len(funds_df)/len(df)*100:.1f}% of full dataset)")
    print(f"{'═'*55}")

    print(f"\nHits by pattern label:")
    for label, count in results_df["matched_pattern_label"].value_counts().items():
        n_review = results_df[results_df["matched_pattern_label"] == label]["needs_review"].sum()
        flag = "  ⚠ needs review" if n_review > 0 else ""
        print(f"  {label:<30} {count:>4}{flag}")

    print(f"\nHits by language group:")
    for lang, count in results_df["matched_language"].value_counts().items():
        print(f"  {lang:<10} {count:>4}")

    review_count = results_df["needs_review"].sum()
    print(f"\nRows flagged for human review: {review_count} "
          f"({review_count/len(results_df)*100:.1f}% of all hits)")

    if "Matched_Via" in results_df.columns:
        exp_hits  = (results_df["Matched_Via"] == "expansion").sum()
        exp_funds = results_df[results_df["Matched_Via"] == "expansion"][ID_COL].nunique()
        print(f"\nOf which, found ONLY via abbreviation expansion (not on the raw name): "
              f"{exp_hits} pattern hits across {exp_funds} funds")



═══════════════════════════════════════════════════════
  IMPACT FUND CANDIDATES: 39 of 6356 funds
  (0.6% of full dataset)
═══════════════════════════════════════════════════════

Hits by pattern label:
  impact                           16
  stewards                          8
  transition                        6  ⚠ needs review
  climate_action                    3  ⚠ needs review
  engagement                        2
  better_world                      1
  sdg                               1
  transformative                    1
  better_future                     1
  sustainable_development           1  ⚠ needs review

Hits by language group:
  EN           40

Rows flagged for human review: 4 (10.0% of all hits)

Of which, found ONLY via abbreviation expansion (not on the raw name): 4 pattern hits across 4 funds


## CELL 8 — Output to Excel

Writes four sheets: **Matches** (one row per hit), **Funds_Deduped** (one row per fund), **Summary** (pattern-level stats), **Token_Scan** (unknown abbreviation candidates).

**The EU notebook's second, more elaborate output block is intentionally omitted here** — a styled workbook with "Overview Stats" / "Term × Language" / "Language Matrix (counts)" / "Language Matrix (rates %)" sheets, built specifically around EU SFDR Article 8/9 classification and cross-language comparison. Neither applies to this dataset: SFDR Article 8/9 doesn't apply to US-domiciled funds (no such column exists here), and a language matrix is degenerate when every match is English — there's no second language to compare against. The simple 4-sheet block below is unchanged from the source and is sufficient for a single-language dataset.

In [9]:
# ============================================================
# Output to Excel -- simple 4-sheet version only.
#
# The source EU notebook's Cell 8 contained a SECOND, more elaborate
# block (styled workbook with "Overview Stats" / "Term x Language" /
# "Language Matrix (counts)" / "Language Matrix (rates %)" sheets)
# built specifically around EU SFDR Article 8/9 classification and
# cross-language comparison. That block is intentionally omitted here:
#   - SFDR Article 8/9 does not apply to US-domiciled funds (no such
#     column exists in this dataset), so the SFDR split sheet has
#     nothing to compute.
#   - A "Language Matrix" is degenerate when every match is English --
#     there is no second language to compare against.
# The simple 4-sheet block below is unchanged from the source and is
# sufficient for a single-language dataset.
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
outfile = OUTPUT_DIR / f"Impact_Fund_Candidates_US_{timestamp}.xlsx"

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    # Sheet 1 -- Full results (one row per pattern hit)
    results_df.to_excel(writer, sheet_name="Matches", index=False)

    # Sheet 2 -- One row per fund (first/most-significant match)
    if len(results_df) > 0:
        deduped = (
            results_df
            .sort_values("needs_review")          # confirmed hits first
            .drop_duplicates(subset=ID_COL, keep="first")
        )
        deduped.to_excel(writer, sheet_name="Funds_Deduped", index=False)

    # Sheet 3 -- Summary statistics
    if len(results_df) > 0:
        summary_rows = []
        for label, grp in results_df.groupby("matched_pattern_label"):
            lang = grp["matched_language"].iloc[0]
            n_funds = grp[ID_COL].nunique()
            n_review = int(grp["needs_review"].sum())
            example = grp[NAME_COL].iloc[0]
            summary_rows.append({
                "Pattern Label":    label,
                "Language":         lang,
                "Funds Matched":    n_funds,
                "Needs Review (n)": n_review,
                "Example Name":     example,
            })
        summary_df = pd.DataFrame(summary_rows).sort_values("Funds Matched", ascending=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

    # Sheet 4 -- Token scan: unknown abbreviations from Cell 4
    abbrev_rows = [
        {"Token": tok, "Count": count, "Known Expansion": ABBREV_EXPANSION.get(tok, "UNKNOWN")}
        for tok, count in unknown_abbrevs[:100]
    ]
    pd.DataFrame(abbrev_rows).to_excel(writer, sheet_name="Token_Scan", index=False)

print(f"\nOutput written to:\n  {outfile}")
print(f"\nSheets: Matches | Funds_Deduped | Summary | Token_Scan")



Output written to:
  /Users/dannyhogan/Desktop/Hogan_RA_Work/Impact_Fund_Candidates_US_20260717_1650.xlsx

Sheets: Matches | Funds_Deduped | Summary | Token_Scan
